# OpenMP Directives: Work-Sharing Constructs

**A work-sharing construct divides the execution of the enclosed code region among the members of the team that encounter it.**

* Work-sharing constructs do not launch new threads

* There is no implied barrier upon entry to a work-sharing construct, however there is an implied barrier at the end of a work sharing construct.


**Restrictions**:
* A work-sharing construct must be enclosed dynamically within a parallel region in order for the directive to execute in parallel.

* Work-sharing constructs must be encountered by all members of a team or none at all.

* Successive work-sharing constructs must be encountered in the same order by all members of a team.

## Types of Work-Sharing Constructs

* **DO / for**
    * Shares iterations of a loop across the team. Represents a type of “data parallelism”.


    ![Directive Sharing Constructor](images/directive_sharing_constructor_do_for.jpg)


* **Sections** 
    * Breaks work into separate, discrete sections. Each section is executed by a thread. Can be used to implement a type of “functional parallelism”.

    ![Directive Sharing Sections](images/directive_sharing_constructor_sections.jpg)

* **Single**
    * SINGLE serializes a section of code

    ![Directive Sharing Singles](images/directive_sharing_constructor_single.jpg)





# 1- Work-Sharing Constructs: DO / for Directive

**Purpose**

The DO / for directive specifies that the iterations of the loop immediately following it must be executed in parallel by the team. This assumes a parallel region has already been initiated, otherwise it executes in serial on a single processor.

**Format**

### Exemple in C/C++

DO / for Directive

**Simple vector-add program**

* Arrays A, B, C, and variable N will be shared by all threads.
* Variable I will be private to each thread; each thread will have its own unique copy.
* The iterations of the loop will be distributed dynamically in CHUNK sized pieces.
* Threads will not synchronize upon completing their individual pieces of work (NOWAIT).

## **Running the example**

**Obs: dentro do notebook é executada apenas uma thread. Por isso, execute fora do notebook para ver o paralelismo.** 
* Aumente o valor de "n", rode no notebook, depois rode fora do notebook. Compare os tempos.

1. Na pasta ompenmp/notebooks/codigo_externo, crie um aquivo chamado: 
    * "sharing_contructs_do_for.cpp".

2. Clique duas vezes para abrí-lo no editor do VSCode.

3. Copie e cole o código acima dentro do arquivo.

4. No terminal Linux, vá até a pasta do aquivo: 
    * cd ompenmp/notebooks/codigo_externo

5. Compile o aquivo:
    * g++ -fopenmp sharing_contructs_do_for.cpp -o sharing_contructs_do_for.exe

6. Execute o programa:
    ./sharing_contructs_do_for.exe

In [13]:
#include "notebooks_reserved_code/openmp_config.h"

#include <omp.h>

int main()
{/* This is a parallelized version of the vector addition example */
  int n = 1000;
  int chunksize = 100;
  int i, chunk;
  float a[n], b[n], c[n];

  /* Some initializations */
  for(i=0; i < n; i++)
    a[i] = b[i] = i * 1.0;

  /* Print the first positions of the vectors a and b */
  for(i=0; i < 5; i++)
    printf("a[%d] = %.1f, b[%d] = %.1f\n", i, a[i], i, b[i]);

  
  double start_time = omp_get_wtime(); // Start time count

  printf("Sum c[i] = a[i] + b[i]\n");
  #pragma omp parallel shared(a,b,c,chunksize) private(i)
  {
    /* Fork a team of threads giving them their own copies of a, b, c */
    #pragma omp for schedule(dynamic,chunksize) nowait
    for(i=0; i < n; i++)
      c[i] = a[i] + b[i];
  }  /* end of parallel section */

  double end_time = omp_get_wtime(); // End time count
  printf("Time taken: %f seconds\n", end_time - start_time);

  /* Print the first positions of the vector c */
  for(i=0; i < 5; i++)
    printf("c[%d] = %.1f\n", i, c[i]);
}
main()

a[0] = 0.0, b[0] = 0.0
a[1] = 1.0, b[1] = 1.0
a[2] = 2.0, b[2] = 2.0
a[3] = 3.0, b[3] = 3.0
a[4] = 4.0, b[4] = 4.0
Sum c[i] = a[i] + b[i]
Time taken: 0.000002 seconds
c[0] = 0.0
c[1] = 2.0
c[2] = 4.0
c[3] = 6.0
c[4] = 8.0


0

### Clauses

1. **SCHEDULE**

Describes how iterations of the loop are divided among the threads in the team. The default schedule is implementation dependent. For a discussion on how one type of scheduling may be more optimal than others, see https://forum.openmp.org/viewtopic.php?t=83.

2. **STATIC**

Loop iterations are divided into pieces of size chunk and then statically assigned to threads. If chunk is not specified, the iterations are evenly (if possible) divided contiguously among the threads.

3. **DYNAMIC**

Loop iterations are divided into pieces of size chunk, and dynamically scheduled among the threads; when a thread finishes one chunk, it is dynamically assigned another. The default chunk size is 1.

4. **GUIDED**

Iterations are dynamically assigned to threads in blocks as threads request them until no blocks remain to be assigned. Similar to DYNAMIC except that the block size decreases each time a parcel of work is given to a thread. 

* The size of the initial block is proportional to:

    * number_of_iterations / number_of_threads

* Subsequent blocks are proportional to

    * number_of_iterations_remaining / number_of_threads

The chunk parameter defines the minimum block size. The default chunk size is 1.

5. **RUNTIME**

The scheduling decision is deferred until runtime by the environment variable OMP_SCHEDULE. It is illegal to specify a chunk size for this clause.

6. **AUTO**

The scheduling decision is delegated to the compiler and/or runtime system.

7. **NO WAIT / nowait**

If specified, then threads do not synchronize at the end of the parallel loop.

8. **ORDERED**

Specifies that the iterations of the loop must be executed as they would be in a serial program.

9. **COLLAPSE**

Specifies how many loops in a nested loop should be collapsed into one large iteration space and divided according to the schedule clause. The sequential execution of the iterations in all associated loops determines the order of the iterations in the collapsed iteration space.

Other clauses are described in detail later, in the Data Scope Attribute Clauses section.

**Restrictions**:

* The DO loop can not be a DO WHILE loop, or a loop without loop control. Also, the loop iteration variable must be an integer and the loop control parameters must be the same for all threads.

* Program correctness must not depend upon which thread executes a particular iteration.

* It is illegal to branch (goto) out of a loop associated with a DO/for directive.

* The chunk size must be specified as a loop invarient integer expression, as there is no synchronization during its evaluation by different threads.

* ORDERED, COLLAPSE and SCHEDULE clauses may appear once each.

* See the OpenMP specification document for additional restrictions.

# 2 - Work-Sharing Constructs: SECTIONS Directive

**Purpose**

The SECTIONS directive is a non-iterative work-sharing construct. It specifies that the enclosed section(s) of code are to be divided among the threads in the team.

Independent SECTION directives are nested within a SECTIONS directive. Each SECTION is executed once by a thread in the team. Different sections may be executed by different threads. It is possible for a thread to execute more than one section if it is quick enough and the implementation permits such.

**Format**

## **Running the example**

**Obs: dentro do notebook é executada apenas uma thread. Por isso, execute fora do notebook para ver o paralelismo.** 
* Aumente o valor de "n", rode no notebook, depois rode fora do notebook. Compare os tempos.

1. Na pasta ompenmp/notebooks/codigo_externo, crie um aquivo chamado: 
    * "sharing_contructs_section.cpp".

2. Clique duas vezes para abrí-lo no editor do VSCode.

3. Copie e cole o código acima dentro do arquivo.

4. No terminal Linux, vá até a pasta do aquivo: 
    * cd ompenmp/notebooks/codigo_externo

5. Compile o aquivo:
    * g++ -fopenmp sharing_contructs_section.cpp -o sharing_contructs_section.exe

6. Execute o programa:
    ./sharing_contructs_section.exe

Simple program demonstrating that different blocks of work will be done by different threads:

In [20]:
#include "notebooks_reserved_code/openmp_config.h"
#include <omp.h>
int main()
{

	int i;
	int n = 1000;
	float a[n], b[n], c[n], d[n];

	/* Some initializations */
	for(i=0; i < n; i++) {
	  a[i] = i * 1.5;
	  b[i] = i + 22.35;
	  }

	#pragma omp parallel shared(a,b,c,d) private(i)
	{

	  #pragma omp sections nowait
	  {

		#pragma omp section
		for(i=0; i < n; i++)
		  c[i] = a[i] + b[i];

		#pragma omp section
		for (i=0; i < n; i++)
		  d[i] = a[i] * b[i];

	   }  /* end of sections */

	}  /* end of parallel section */
	  /* Print the first positions of the vectors a and b */
	for(i=0; i < 5; i++)
	printf("a[%d] = %.1f, b[%d] = %.1f\n", i, a[i], i, b[i]);
	
	/* Print the first positions of the vector c */
	for(i=0; i < 5; i++)
	printf("c[%d] = %.1f\n", i, c[i]);

	/* Print the first positions of the vector c */
	for(i=0; i < 5; i++)
	printf("d[%d] = %.1f\n", i, d[i]);

}
main()

a[0] = 0.0, b[0] = 22.4
a[1] = 1.5, b[1] = 23.4
a[2] = 3.0, b[2] = 24.4
a[3] = 4.5, b[3] = 25.4
a[4] = 6.0, b[4] = 26.4
c[0] = 22.4
c[1] = 24.9
c[2] = 27.4
c[3] = 29.9
c[4] = 32.3
d[0] = 0.0
d[1] = 35.0
d[2] = 73.1
d[3] = 114.1
d[4] = 158.1


0

**Clauses**

There is an implied barrier at the end of a SECTIONS directive, unless the NOWAIT/nowait clause is used. Clauses are described in detail later, in the Data Scope Attribute Clauses section.

**Questions**

* What happens if the number of threads and the number of SECTIONs are different? 
* More threads than SECTIONs? Less threads than SECTIONs?
* Which thread executes which SECTION?

**Restrictions**

* It is illegal to branch (goto) into or out of section blocks.
* SECTION directives must occur within the lexical extent of an enclosing SECTIONS directive (no orphan SECTIONs).

# 3 - Work-Sharing Constructs: Work-Sharing Constructs: SINGLE Directive

**Purpose**

The SINGLE directive specifies that the enclosed code is to be executed by only one thread in the team.

May be useful when dealing with sections of code that are not thread safe (such as I/O)

**Format**

**Clauses**

Threads in the team that do not execute the SINGLE directive, wait at the end of the enclosed code block, unless a NOWAIT/nowait clause is specified.

Clauses are described in detail later, in the Data Scope Attribute Clauses section.

**Restrictions**

It is illegal to branch into or out of a SINGLE block.